# 06 · 2축 휨 상관도

원 문서의 `biaxial_bending.ipynb` 에 대응한다. 계수 축력이 주어졌을 때
중립축 각도를 한 바퀴 돌리며 각 방향의 설계 휨강도를 구한다.

강도감소계수는 방향마다 달라진다. 같은 축력이라도 중립축 방향에 따라
최외단 인장철근의 순인장변형률이 달라지기 때문이다
(KDS 14 20 10 4.3.3(2)).

In [1]:
import matplotlib.pyplot as plt
import numpy as np

# 한글 글꼴이 없는 환경에서도 그림이 깨지지 않도록 축 라벨은 ASCII 로 둔다
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 96

In [2]:
from concreteproperties import ConcreteSection
from sectionproperties.pre.library import concrete_rectangular_section

from concreteproperties_kds import KDS


def beam_section(fck=27, fy=400):
    """400 x 600 보 단면 (상부 2-D16, 하부 4-D22, 피복 50 mm)."""
    kds = KDS(column_type="tie")
    conc = kds.create_concrete_material(compressive_strength=fck)
    steel = kds.create_steel_material(yield_strength=fy)

    geom = concrete_rectangular_section(
        d=600, b=400,
        dia_top=16, area_top=198.6, n_top=2, c_top=50,
        dia_bot=22, area_bot=387.1, n_bot=4, c_bot=50,
        n_circle=16, conc_mat=conc, steel_mat=steel,
    )
    conc_sec = ConcreteSection(geom)
    kds.assign_concrete_section(conc_sec)
    return kds, conc_sec


def column_section(fck=27, fy=400, column_type="tie"):
    """500 x 500 기둥 단면 (8-D22, 피복 50 mm)."""
    kds = KDS(column_type=column_type)
    conc = kds.create_concrete_material(compressive_strength=fck)
    steel = kds.create_steel_material(yield_strength=fy)

    geom = concrete_rectangular_section(
        d=500, b=500,
        dia_top=22, area_top=387.1, n_top=3, c_top=50,
        dia_bot=22, area_bot=387.1, n_bot=3, c_bot=50,
        dia_side=22, area_side=387.1, n_side=1, c_side=50,
        n_circle=16, conc_mat=conc, steel_mat=steel,
    )
    conc_sec = ConcreteSection(geom)
    kds.assign_concrete_section(conc_sec)
    return kds, conc_sec

In [3]:
kds, _ = column_section()

n_design = 1200e3
f_bb, phis = kds.biaxial_bending_diagram(
    n_design=n_design, n_points=32, progress_bar=False
)

m_x = np.array([r.m_x for r in f_bb.results]) / 1e6
m_y = np.array([r.m_y for r in f_bb.results]) / 1e6

print(f"Nd = {n_design / 1e3:,.0f} kN")
print(f"phi 범위 : {min(phis):.3f} ~ {max(phis):.3f}")
print(f"1축 휨강도 phi*Mnx = {np.max(np.abs(m_x)):.1f} kN.m")
print(f"1축 휨강도 phi*Mny = {np.max(np.abs(m_y)):.1f} kN.m")

Nd = 1,200 kN
phi 범위 : 0.688 ~ 0.828
1축 휨강도 phi*Mnx = 386.2 kN.m
1축 휨강도 phi*Mny = 386.2 kN.m


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))

axes[0].plot(m_x, m_y, "-o", ms=3)
axes[0].set_xlabel("phi*Mx (kN.m)")
axes[0].set_ylabel("phi*My (kN.m)")
axes[0].set_title(f"Biaxial bending, Nd = {n_design / 1e3:,.0f} kN")
axes[0].set_aspect("equal")
axes[0].grid(alpha=0.3)

theta = np.degrees([r.theta for r in f_bb.results])
axes[1].plot(theta, phis, "-o", ms=3)
axes[1].set_xlabel("theta (deg)")
axes[1].set_ylabel("phi")
axes[1].set_title("Strength reduction factor by direction")
axes[1].grid(alpha=0.3)
fig.tight_layout()

정사각형 대칭 단면이라 상관면이 네 방향으로 대칭이다. 45° 방향에서 강도가
가장 작고, $\phi$ 도 가장 낮다.

## 축력에 따른 상관면 변화

In [5]:
fig, ax = plt.subplots(figsize=(6, 5.4))

for n_d in [0, 800e3, 1600e3, 2400e3]:
    bb, _ = kds.biaxial_bending_diagram(
        n_design=n_d, n_points=24, progress_bar=False
    )
    ax.plot(
        np.array([r.m_x for r in bb.results]) / 1e6,
        np.array([r.m_y for r in bb.results]) / 1e6,
        label=f"Nd = {n_d / 1e3:,.0f} kN",
    )

ax.set_xlabel("phi*Mx (kN.m)")
ax.set_ylabel("phi*My (kN.m)")
ax.set_title("Biaxial bending diagrams")
ax.set_aspect("equal")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

축력이 균형점 부근일 때 상관면이 가장 크다.